In [ ]:
# Bengali Character AutoML for Google Colab
# Upload a zip file with folders like:
# dataset.zip
#   train/
#     অ/
#       image1.bmp
#     আ/
#       image2.bmp
#   test/
#     অ/
#       image3.bmp
#     আ/
#       image4.bmp

import os
import time
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from google.colab import files
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    Perceptron,
    RidgeClassifier,
    SGDClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import BernoulliNB, ComplementNB, GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid, RadiusNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import LinearSVC, NuSVC, SVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier


ZIP_LIMIT_MB = 500
DATA_DIR = Path("/content/bengali_character_data")
IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}

# Tune these for speed/accuracy.
IMAGE_SIZE = 28
MAX_TRAIN_IMAGES = 30000
MAX_TEST_IMAGES = 10000
MAX_MODELS = 24
RANDOM_STATE = 42
MAX_ITER = 800
INVERT_COLORS = False
APPLY_THRESHOLD = False


def upload_and_extract_zip() -> Path:
    print("Upload your dataset zip file now...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")

    zip_name = next(iter(uploaded.keys()))
    zip_path = Path("/content") / zip_name
    size_mb = zip_path.stat().st_size / (1024 * 1024)
    if size_mb > ZIP_LIMIT_MB:
        raise ValueError(f"Zip is {size_mb:.1f} MB. Limit is {ZIP_LIMIT_MB} MB.")
    if zip_path.suffix.lower() != ".zip":
        raise ValueError("Please upload a .zip file.")

    if DATA_DIR.exists():
        for child in DATA_DIR.iterdir():
            if child.is_file():
                child.unlink()
            else:
                import shutil

                shutil.rmtree(child)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = DATA_DIR / member.filename
            if not str(target.resolve()).startswith(str(DATA_DIR.resolve())):
                raise ValueError(f"Unsafe zip path found: {member.filename}")
        archive.extractall(DATA_DIR)

In [ ]:
"""
Bengali Character AutoML for Google Colab
=========================================
This script implements a high-performance AutoML pipeline tailored for Google Colab
to classify handwritten Bengali characters. It processes uploaded datasets, trains
20+ machine learning models, evaluates their performance, and outputs a stunning,
highly interactive, state-of-the-art HTML report containing interactive charts and
detailed tables with vanilla JavaScript filtering, search, and sorting.

Upload a zip file structured as:
dataset.zip
  train/
    অ/
      image1.png
    আ/
      image2.png
  test/
    অ/
      image3.png
    আ/
      image4.png
"""

import os
import time
import zipfile
import base64
import json
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    Perceptron,
    RidgeClassifier,
    SGDClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.naive_bayes import BernoulliNB, ComplementNB, GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

# Google Colab specific uploads/downloads
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- CONFIGURABLE HYPERPARAMETERS ---
ZIP_LIMIT_MB = 500
DATA_DIR = Path("./bengali_character_data") if not IN_COLAB else Path("/content/bengali_character_data")
IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}

# Preprocessing & Constraints
IMAGE_SIZE = 28  # Resize dimensions (e.g., 28x28 = 784 features)
MAX_TRAIN_IMAGES = 30000
MAX_TEST_IMAGES = 10000
MAX_ITER = 1000
RANDOM_STATE = 42
INVERT_COLORS = False
APPLY_THRESHOLD = False

# PCA components reduction to speed up training of heavy models
USE_PCA = True
PCA_COMPONENTS = 0.95  # Retain 95% variance


def upload_and_extract_zip() -> Path:
    """Handles the uploading and extraction of the zip file in Colab or locally."""
    if IN_COLAB:
        print("💡 [STEP 1] Upload your dataset zip file...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("❌ No file uploaded.")
        zip_name = next(iter(uploaded.keys()))
        zip_path = Path("/content") / zip_name
    else:
        print("💡 [STEP 1] Running locally. Looking for dataset.zip in current directory...")
        zip_path = Path("dataset.zip")
        if not zip_path.exists():
            zips = list(Path(".").glob("*.zip"))
            if zips:
                zip_path = zips[0]
                print(f"Found zip: {zip_path}")
            else:
                raise FileNotFoundError("❌ Please place a 'dataset.zip' file in the current directory to run locally.")

    size_mb = zip_path.stat().st_size / (1024 * 1024)
    if size_mb > ZIP_LIMIT_MB:
        raise ValueError(f"❌ Zip file size ({size_mb:.1f} MB) exceeds the limit of {ZIP_LIMIT_MB} MB.")

    print(f"🔄 Extracting dataset to {DATA_DIR.resolve()}...")
    if DATA_DIR.exists():
        import shutil
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            # Guard against Zip Slip vulnerability
            target = (DATA_DIR / member.filename).resolve()
            if not str(target).startswith(str(DATA_DIR.resolve())):
                raise ValueError(f"❌ Unsafe zip path found: {member.filename}")
        archive.extractall(DATA_DIR)
    print("✅ Extraction complete.")
    return DATA_DIR


def load_dataset(data_path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """Loads and preprocesses images from train/ and test/ folders inside data_path."""
    print("💡 [STEP 2] Loading and preprocessing images...")

    # Identify directories (handle nested structures if zip folder was wrapped)
    train_dir = data_path / "train"
    test_dir = data_path / "test"

    if not train_dir.exists() or not test_dir.exists():
        subdirs = [d for d in data_path.iterdir() if d.is_dir()]
        for subdir in subdirs:
            if (subdir / "train").exists() and (subdir / "test").exists():
                train_dir = subdir / "train"
                test_dir = subdir / "test"
                break
        else:
            raise FileNotFoundError(
                "❌ Could not locate 'train' and 'test' folders inside the zip file.\n"
                "Please ensure structure is: zip -> train/ & test/ -> label folders (e.g., অ/, আ/) -> images"
            )

    # Get sorted list of class names (from train directories)
    class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    print(f"🏷️ Found {len(class_names)} classes: {class_names}")

    def process_folder(folder_path: Path, max_images: int) -> Tuple[np.ndarray, np.ndarray]:
        images_loaded = 0
        X_list, y_list = [], []

        # Walk through sorted class folders
        for class_name in class_names:
            class_folder = folder_path / class_name
            if not class_folder.exists():
                continue

            idx = class_to_idx[class_name]
            files_in_folder = [
                f for f in class_folder.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            ]

            for file_path in files_in_folder:
                if images_loaded >= max_images:
                    break
                try:
                    with Image.open(file_path) as img:
                        img = img.convert("L")
                        if INVERT_COLORS:
                            img = ImageOps.invert(img)
                        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
                        if APPLY_THRESHOLD:
                            img = img.point(lambda p: 255 if p > 127 else 0)

                        arr = np.array(img, dtype=np.float32) / 255.0
                        X_list.append(arr.flatten())
                        y_list.append(idx)
                        images_loaded += 1
                except Exception:
                    pass

            if images_loaded >= max_images:
                break

        if len(X_list) == 0:
            raise ValueError(f"No valid images loaded from directory: {folder_path}")

        return np.array(X_list), np.array(y_list)

    print(f"⏳ Reading train folder (up to {MAX_TRAIN_IMAGES} images)...")
    X_train, y_train = process_folder(train_dir, MAX_TRAIN_IMAGES)
    print(f"⏳ Reading test folder (up to {MAX_TEST_IMAGES} images)...")
    X_test, y_test = process_folder(test_dir, MAX_TEST_IMAGES)

    print(f"📊 Loaded Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
    return X_train, y_train, X_test, y_test, class_names


def get_models() -> Dict[str, Any]:
    """Defines 20+ scikit-learn models covering multiple ML families."""
    return {
        "Logistic Regression": LogisticRegression(max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "Ridge Classifier": RidgeClassifier(random_state=RANDOM_STATE),
        "SGD Classifier": SGDClassifier(max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "Perceptron": Perceptron(max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "Passive Aggressive": PassiveAggressiveClassifier(max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "Linear SVC": LinearSVC(max_iter=MAX_ITER, random_state=RANDOM_STATE, dual="auto"),
        "Support Vector Classifier (SVC)": SVC(kernel="rbf", random_state=RANDOM_STATE),
        "K-Neighbors (K=3)": KNeighborsClassifier(n_neighbors=3),
        "K-Neighbors (K=5)": KNeighborsClassifier(n_neighbors=5),
        "Nearest Centroid": NearestCentroid(),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Extra Tree": ExtraTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, random_state=RANDOM_STATE),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE),
        "Bagging Classifier": BaggingClassifier(n_estimators=10, random_state=RANDOM_STATE),
        "Gaussian Naive Bayes": GaussianNB(),
        "Bernoulli Naive Bayes": BernoulliNB(),
        "Complement Naive Bayes": ComplementNB(),
        "Multinomial Naive Bayes": MultinomialNB(),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
        "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(),
        "Dummy Baseline": DummyClassifier(strategy="most_frequent")
    }


def generate_html_report(results: List[Dict[str, Any]], class_names: List[str]) -> str:
    """Generates a glassmorphic dark-theme HTML report with interactive Chart.js visualizations."""
    sorted_results = sorted(results, key=lambda x: x["accuracy"], reverse=True)
    chart_labels = [r["name"] for r in sorted_results]
    accuracies = [round(r["accuracy"] * 100, 2) for r in sorted_results]
    train_times = [round(r["train_time"], 4) for r in sorted_results]
    f1_scores = [round(r["f1_weighted"] * 100, 2) for r in sorted_results]

    detailed_table_rows = []
    for rank, r in enumerate(sorted_results, 1):
        row = f"""
        <tr class="hover:bg-slate-800/40 border-b border-slate-700/50 transition-colors">
            <td class="px-6 py-4 whitespace-nowrap text-sm font-semibold text-teal-400">#{rank}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-slate-200 font-bold">{r['name']}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-emerald-400 font-extrabold">{r['accuracy']:.4f}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-sky-400 font-bold">{r['f1_weighted']:.4f}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-indigo-400">{r['precision_weighted']:.4f}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-purple-400">{r['recall_weighted']:.4f}</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-amber-400 font-medium">{r['train_time']:.4f}s</td>
            <td class="px-6 py-4 whitespace-nowrap text-sm text-slate-400">{r['predict_time']:.4f}s</td>
            <td class="px-6 py-4 whitespace-nowrap text-right text-sm font-medium">
                <button onclick="showDetails('{r['name']}')" class="text-teal-400 hover:text-teal-300 bg-teal-400/10 hover:bg-teal-400/20 px-3 py-1.5 rounded-lg border border-teal-500/30 transition-all">Details</button>
            </td>
        </tr>
        """
        detailed_table_rows.append(row)

    detailed_table_html = "\n".join(detailed_table_rows)

    js_results_data = {}
    for r in results:
        report_cleaned = "<br>".join(r["class_report"].split("\n"))
        js_results_data[r["name"]] = {
            "name": r["name"],
            "accuracy": round(r["accuracy"], 5),
            "train_time": round(r["train_time"], 4),
            "predict_time": round(r["predict_time"], 4),
            "class_report": report_cleaned,
            "confusion_matrix": r["conf_matrix"].tolist()
        }

    js_data_json = json.dumps(js_results_data)
    class_names_json = json.dumps(class_names)

    html_content = f"""<!DOCTYPE html>
<html lang="en" class="dark">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Bengali Character AutoML - Benchmark Dashboard</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;700&display=swap" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script>
        tailwind.config = {{
            darkMode: 'class',
            theme: {{
                extend: {{
                    fontFamily: {{
                        sans: ['Outfit', 'sans-serif'],
                        mono: ['JetBrains Mono', 'monospace'],
                    }}
                }}
            }}
        }}
    </script>
    <style>
        body {{
            background: linear-gradient(135deg, #0f172a 0%, #1e1b4b 100%);
            font-family: 'Outfit', sans-serif;
            color: #f1f5f9;
            min-height: 100vh;
        }}
        .glass-panel {{
            background: rgba(30, 41, 59, 0.45);
            backdrop-filter: blur(16px);
            -webkit-backdrop-filter: blur(16px);
            border: 1px solid rgba(255, 255, 255, 0.08);
            box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);
        }}
        .glass-card {{
            background: rgba(30, 41, 59, 0.6);
            border: 1px solid rgba(255, 255, 255, 0.05);
            transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
        }}
        .glass-card:hover {{
            transform: translateY(-4px);
            border-color: rgba(20, 184, 166, 0.3);
            box-shadow: 0 12px 40px -10px rgba(20, 184, 166, 0.2);
        }}
        ::-webkit-scrollbar {{
            width: 8px;
            height: 8px;
        }}
        ::-webkit-scrollbar-track {{
            background: #0f172a;
        }}
        ::-webkit-scrollbar-thumb {{
            background: #334155;
            border-radius: 9999px;
        }}
        ::-webkit-scrollbar-thumb:hover {{
            background: #475569;
        }}
    </style>
</head>
<body class="p-6 md:p-12">
    <header class="max-w-7xl mx-auto mb-10 text-center md:text-left flex flex-col md:flex-row justify-between items-center gap-6 pb-8 border-b border-slate-800/80">
        <div>
            <div class="inline-flex items-center gap-2 px-3 py-1.5 rounded-full bg-teal-500/10 border border-teal-500/30 text-teal-400 text-sm font-semibold mb-3">
                <span class="w-2.5 h-2.5 rounded-full bg-teal-400 animate-pulse"></span>
                Bengali AutoML Performance Engine
            </div>
            <h1 class="text-4xl md:text-5xl font-extrabold tracking-tight text-white">
                Handwritten Character <span class="bg-gradient-to-r from-teal-400 via-sky-400 to-indigo-500 bg-clip-text text-transparent">AutoML Dashboard</span>
            </h1>
            <p class="mt-2 text-slate-400 text-base md:text-lg max-w-2xl font-light">
                Comprehensive benchmarking report of 20+ Machine Learning classifiers evaluated on Bengali handwritten symbols.
            </p>
        </div>
        <div class="flex gap-4">
            <div class="glass-panel px-6 py-4 rounded-2xl flex flex-col items-center">
                <span class="text-xs font-semibold text-slate-400 uppercase tracking-widest">Total Models</span>
                <span class="text-3xl font-extrabold text-teal-400 mt-1">{len(results)}</span>
            </div>
            <div class="glass-panel px-6 py-4 rounded-2xl flex flex-col items-center">
                <span class="text-xs font-semibold text-slate-400 uppercase tracking-widest">Best Accuracy</span>
                <span class="text-3xl font-extrabold text-emerald-400 mt-1">{sorted_results[0]['accuracy']*100:.2f}%</span>
            </div>
        </div>
    </header>

    <main class="max-w-7xl mx-auto space-y-10">
        <section class="grid grid-cols-1 lg:grid-cols-2 gap-8">
            <div class="glass-panel p-6 rounded-2xl">
                <h3 class="text-lg font-bold text-slate-200 mb-4 flex items-center gap-2">📊 Classifier Accuracy Benchmarking (%)</h3>
                <div class="relative w-full h-[400px]">
                    <canvas id="accuracyChart"></canvas>
                </div>
            </div>
            <div class="glass-panel p-6 rounded-2xl">
                <h3 class="text-lg font-bold text-slate-200 mb-4 flex items-center gap-2">⚡ Accuracy vs. Training Speed (s)</h3>
                <div class="relative w-full h-[400px]">
                    <canvas id="speedChart"></canvas>
                </div>
            </div>
        </section>

        <section class="glass-panel rounded-2xl overflow-hidden">
            <div class="p-6 border-b border-slate-800 flex flex-col md:flex-row justify-between items-start md:items-center gap-4">
                <div>
                    <h3 class="text-xl font-bold text-white">🏆 Benchmark Leaderboard</h3>
                    <p class="text-slate-400 text-sm mt-1">Detailed performance metrics of all trained classifiers sorted by validation accuracy.</p>
                </div>
                <div class="relative w-full md:w-72">
                    <input type="text" id="searchInput" onkeyup="filterTable()" placeholder="Search models..." class="w-full bg-slate-900/60 border border-slate-700/60 rounded-xl px-4 py-2.5 text-sm text-slate-200 focus:outline-none focus:border-teal-500/60 placeholder-slate-500">
                </div>
            </div>

            <div class="overflow-x-auto">
                <table class="min-w-full divide-y divide-slate-800" id="benchmarkTable">
                    <thead class="bg-slate-900/40">
                        <tr>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider">Rank</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider">Classifier Name</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider cursor-pointer" onclick="sortTable(2)">Accuracy ↕</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider cursor-pointer" onclick="sortTable(3)">Weighted F1 ↕</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider">Precision</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider">Recall</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider cursor-pointer" onclick="sortTable(6)">Train Time ↕</th>
                            <th scope="col" class="px-6 py-3 text-left text-xs font-bold text-slate-400 uppercase tracking-wider">Predict Time</th>
                            <th scope="col" class="relative px-6 py-3"><span class="sr-only">Actions</span></th>
                        </tr>
                    </thead>
                    <tbody class="divide-y divide-slate-800/40 bg-slate-900/10">
                        {detailed_table_html}
                    </tbody>
                </table>
            </div>
        </section>

        <section id="inspectorPanel" class="glass-panel p-8 rounded-2xl hidden transition-all duration-300">
            <div class="flex justify-between items-start border-b border-slate-800 pb-4 mb-6">
                <div>
                    <h3 class="text-2xl font-bold text-teal-400" id="inspectorName">Model Name</h3>
                    <p class="text-slate-400 text-sm mt-1">Deep inspection of classification reports and error matrices.</p>
                </div>
                <button onclick="closeInspector()" class="text-slate-400 hover:text-white bg-slate-800 px-3 py-1.5 rounded-lg border border-slate-700/40 text-sm">Close</button>
            </div>

            <div class="grid grid-cols-1 lg:grid-cols-2 gap-8">
                <div>
                    <h4 class="text-base font-bold text-slate-200 mb-3 flex items-center gap-2">📄 Classification Report</h4>
                    <pre class="bg-slate-950/80 border border-slate-800 rounded-xl p-5 overflow-auto font-mono text-sm text-slate-300 shadow-inner h-[400px]" id="inspectorReport"></pre>
                </div>
                <div>
                    <h4 class="text-base font-bold text-slate-200 mb-3 flex items-center gap-2">🧭 Confusion Matrix Heatmap</h4>
                    <div class="bg-slate-950/40 border border-slate-800 rounded-xl p-6 h-[400px] overflow-auto flex flex-col items-center justify-center">
                        <div id="matrixContainer" class="grid gap-1 w-full max-w-md aspect-square"></div>
                    </div>
                </div>
            </div>
        </section>
    </main>

    <footer class="max-w-7xl mx-auto mt-16 text-center text-slate-500 text-sm border-t border-slate-800/40 pt-8">
        🤖 Bengali AutoML Engine • Powered by Scikit-Learn & Google Colab.
    </footer>

    <script>
        const modelData = {js_data_json};
        const classNames = {class_names_json};

        const accCtx = document.getElementById('accuracyChart').getContext('2d');
        new Chart(accCtx, {{
            type: 'bar',
            data: {{
                labels: {chart_labels},
                datasets: [{{
                    label: 'Accuracy (%)',
                    data: {accuracies},
                    backgroundColor: 'rgba(20, 184, 166, 0.65)',
                    borderColor: 'rgba(20, 184, 166, 1)',
                    borderWidth: 1.5,
                    borderRadius: 6,
                    hoverBackgroundColor: 'rgba(20, 184, 166, 0.85)'
                }}, {{
                    label: 'F1 Score (%)',
                    data: {f1_scores},
                    backgroundColor: 'rgba(99, 102, 241, 0.4)',
                    borderColor: 'rgba(99, 102, 241, 1)',
                    borderWidth: 1.5,
                    borderRadius: 6,
                    hoverBackgroundColor: 'rgba(99, 102, 241, 0.6)'
                }}]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                scales: {{
                    y: {{
                        grid: {{ color: 'rgba(255,255,255,0.04)' }},
                        ticks: {{ color: '#94a3b8' }},
                        min: 0,
                        max: 100
                    }},
                    x: {{
                        grid: {{ display: false }},
                        ticks: {{ color: '#94a3b8', font: {{ size: 10 }} }}
                    }}
                }},
                plugins: {{
                    legend: {{
                        position: 'top',
                        labels: {{ color: '#f1f5f9', font: {{ family: 'Outfit' }} }}
                    }}
                }}
            }}
        }});

        const speedCtx = document.getElementById('speedChart').getContext('2d');
        const scatterData = {chart_labels}.map((label, idx) => ({{
            x: {train_times}[idx],
            y: {accuracies}[idx],
            label: label
        }}));

        new Chart(speedCtx, {{
            type: 'scatter',
            data: {{
                datasets: [{{
                    label: 'Classifiers',
                    data: scatterData,
                    backgroundColor: 'rgba(56, 189, 248, 0.75)',
                    borderColor: 'rgba(56, 189, 248, 1)',
                    pointRadius: 8,
                    pointHoverRadius: 10
                }}]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                scales: {{
                    y: {{
                        title: {{ display: true, text: 'Accuracy (%)', color: '#94a3b8' }},
                        grid: {{ color: 'rgba(255,255,255,0.04)' }},
                        ticks: {{ color: '#94a3b8' }}
                    }},
                    x: {{
                        title: {{ display: true, text: 'Training Time (seconds)', color: '#94a3b8' }},
                        grid: {{ color: 'rgba(255,255,255,0.04)' }},
                        ticks: {{ color: '#94a3b8' }},
                        type: 'linear'
                    }}
                }},
                plugins: {{
                    legend: {{ display: false }},
                    tooltip: {{
                        callbacks: {{
                            label: function(ctx) {{
                                const pt = ctx.raw;
                                return `${{pt.label}}: Acc=${{pt.y}}%, Time=${{pt.x}}s`;
                            }}
                        }}
                    }}
                }}
            }}
        }});

        function filterTable() {{
            const input = document.getElementById('searchInput');
            const filter = input.value.toLowerCase();
            const table = document.getElementById('benchmarkTable');
            const trs = table.getElementsByTagName('tr');
            for (let i = 1; i < trs.length; i++) {{
                const td = trs[i].getElementsByTagName('td')[1];
                if (td) {{
                    const textVal = td.textContent || td.innerText;
                    trs[i].style.display = textVal.toLowerCase().indexOf(filter) > -1 ? "" : "none";
                }}
            }}
        }}

        let sortDirections = {{}};
        function sortTable(colIndex) {{
            const table = document.getElementById("benchmarkTable");
            let switching = true;
            let dir = sortDirections[colIndex] === "asc" ? "desc" : "asc";
            sortDirections[colIndex] = dir;
            while (switching) {{
                switching = false;
                let rows = table.rows;
                for (var i = 1; i < (rows.length - 1); i++) {{
                    let shouldSwitch = false;
                    let x = rows[i].getElementsByTagName("TD")[colIndex];
                    let y = rows[i+1].getElementsByTagName("TD")[colIndex];
                    let valX = parseFloat(x.innerText) || x.innerText.toLowerCase();
                    let valY = parseFloat(y.innerText) || y.innerText.toLowerCase();
                    if (dir === "asc" && valX > valY) {{ shouldSwitch = true; break; }}
                    else if (dir === "desc" && valX < valY) {{ shouldSwitch = true; break; }}
                }}
                if (shouldSwitch) {{
                    rows[i].parentNode.insertBefore(rows[i + 1], rows[i]);
                    switching = true;
                }}
            }}
        }}

        function showDetails(modelName) {{
            const m = modelData[modelName];
            document.getElementById('inspectorName').innerText = m.name;
            document.getElementById('inspectorReport').innerHTML = m.class_report;

            const container = document.getElementById('matrixContainer');
            container.innerHTML = "";
            const matrix = m.confusion_matrix;
            const size = matrix.length;
            container.style.gridTemplateColumns = `repeat(${{size}}, minmax(0, 1fr))`;

            let maxVal = 1;
            matrix.forEach(row => row.forEach(val => {{ if(val > maxVal) maxVal = val; }}));

            for (let i = 0; i < size; i++) {{
                for (let j = 0; j < size; j++) {{
                    const val = matrix[i][j];
                    const ratio = val / maxVal;
                    const cell = document.createElement('div');
                    cell.className = "flex flex-col items-center justify-center p-1 font-mono text-xs rounded border border-slate-900/50 transition-colors";
                    cell.style.backgroundColor = `rgba(20, 184, 166, ${{Math.max(0.05, ratio * 0.8)}})`;
                    cell.style.color = ratio > 0.45 ? "#ffffff" : "#cbd5e1";
                    cell.title = `Actual: ${{classNames[i]}} -> Predicted: ${{classNames[j]}} (${{val}})`;

                    const labelSpan = document.createElement('span');
                    labelSpan.className = "font-bold text-sm";
                    labelSpan.innerText = val;
                    cell.appendChild(labelSpan);
                    container.appendChild(cell);
                }}
            }}
            const panel = document.getElementById('inspectorPanel');
            panel.classList.remove('hidden');
            panel.scrollIntoView({{ behavior: 'smooth' }});
        }}

        function closeInspector() {{
            document.getElementById('inspectorPanel').classList.add('hidden');
        }}
    </script>
</body>
</html>
"""
    return html_content


def main():
    print("==================================================")
    print("🇧🇩 Bengali Handwritten Character AutoML Benchmarker")
    print("==================================================")

    try:
        data_path = upload_and_extract_zip()
    except Exception as e:
        print(f"\n❌ Error during file upload/extraction: {e}")
        return

    try:
        X_train, y_train, X_test, y_test, class_names = load_dataset(data_path)
    except Exception as e:
        print(f"\n❌ Error during dataset loading: {e}")
        return

    print("\n🔄 Scaling features using MinMaxScaler...")
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if USE_PCA:
        print(f"🧩 Applying PCA to retain {PCA_COMPONENTS * 100}% variance...")
        pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
        X_train_final = pca.fit_transform(X_train_scaled)
        X_test_final = pca.transform(X_test_scaled)
        print(f"📊 Features reduced from {X_train_scaled.shape[1]} to {X_train_final.shape[1]} principal components.")
    else:
        X_train_final = X_train_scaled
        X_test_final = X_test_scaled

    models = get_models()
    results = []

    print("\n🚀 Starting AutoML Benchmarking over 20+ Classifiers...")
    print("--------------------------------------------------------------------------------")
    print(f"{'Classifier Name':<35} | {'Accuracy':<10} | {'Train Time (s)':<15}")
    print("--------------------------------------------------------------------------------")

    for name, model in models.items():
        try:
            clf = clone(model)

            start_train = time.perf_counter()
            clf.fit(X_train_final, y_train)
            train_time = time.perf_counter() - start_train

            start_pred = time.perf_counter()
            y_pred = clf.predict(X_test_final)
            predict_time = time.perf_counter() - start_pred

            acc = accuracy_score(y_test, y_pred)
            precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
            conf_m = confusion_matrix(y_test, y_pred)
            class_rep = classification_report(y_test, y_pred, target_names=class_names, zero_division=0)

            results.append({
                "name": name,
                "accuracy": float(acc),
                "precision_weighted": float(precision),
                "recall_weighted": float(recall),
                "f1_weighted": float(f1),
                "train_time": float(train_time),
                "predict_time": float(predict_time),
                "conf_matrix": conf_m,
                "class_report": class_rep
            })

            print(f"{name:<35} | {acc * 100:>8.2f}% | {train_time:>13.4f}s")

        except Exception as e:
            print(f"⚠️ Failed to evaluate model '{name}': {e}")
            continue

    if not results:
        print("\n❌ Error: All classifiers failed to run. Check data scaling or formats.")
        return

    html_report = generate_html_report(results, class_names)
    report_filename = "bengali_automl_report.html"

    with open(report_filename, "w", encoding="utf-8") as f:
        f.write(html_report)

    print("\n🏆 AutoML benchmark complete!")
    print(f"📁 Dashboard saved successfully to: '{report_filename}'")

    if IN_COLAB:
        print("📥 Downloading dashboard HTML file to your computer...")
        files.download(report_filename)
        print("✅ Finished.")
    else:
        print("💡 Open 'bengali_automl_report.html' in your browser to inspect the results.")


if __name__ == "__main__":
    main()

🇧🇩 Bengali Handwritten Character AutoML Benchmarker
💡 [STEP 1] Upload your dataset zip file...


Saving Dataset.zip to Dataset (1).zip
🔄 Extracting dataset to /content/bengali_character_data...
✅ Extraction complete.
💡 [STEP 2] Loading and preprocessing images...
🏷️ Found 50 classes: ['172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221']
⏳ Reading train folder (up to 30000 images)...
⏳ Reading test folder (up to 10000 images)...
📊 Loaded Train: 12000 samples | Test: 3000 samples

🔄 Scaling features using MinMaxScaler...
🧩 Applying PCA to retain 95.0% variance...
📊 Features reduced from 784 to 261 principal components.

🚀 Starting AutoML Benchmarking over 20+ Classifiers...
--------------------------------------------------------------------------------
Classifier Name                     | 

KeyboardInterrupt: 

In [1]:
"""
Bengali Character AutoML - safe timed version
=============================================

This version keeps the original Colab-friendly workflow, but prevents long model
runs from hanging the notebook/script. Every classifier is evaluated in a
separate process with a hard wall-clock timeout below two minutes.

Expected zip structure:
dataset.zip
  train/
    অ/
      image1.png
  test/
    অ/
      image2.png
"""

from __future__ import annotations

import html
import json
import multiprocessing as mp
import os
import queue
import shutil
import time
import traceback
import zipfile
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
from PIL import Image, ImageOps
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    Perceptron,
    RidgeClassifier,
    SGDClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.naive_bayes import BernoulliNB, ComplementNB, GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

try:
    from google.colab import files

    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False


# --------------------------- Runtime limits ----------------------------
ZIP_LIMIT_MB = 500
MODEL_TIMEOUT_SECONDS = 110  # safely below two minutes, including fit + predict
MAX_TRAIN_IMAGES = 12000
MAX_TEST_IMAGES = 4000
MAX_ITER = 300
RANDOM_STATE = 42

# ------------------------- Preprocessing setup -------------------------
DATA_DIR = Path("/content/bengali_character_data") if IN_COLAB else Path("./bengali_character_data")
IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}
IMAGE_SIZE = 28
INVERT_COLORS = False
APPLY_THRESHOLD = False
USE_PCA = True
PCA_COMPONENTS = 0.95


def upload_and_extract_zip() -> Path:
    """Upload or discover a dataset zip, then extract it safely."""
    if IN_COLAB:
        print("[STEP 1] Upload your dataset zip file...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        zip_path = Path("/content") / next(iter(uploaded.keys()))
    else:
        print("[STEP 1] Running locally. Looking for dataset.zip...")
        zip_path = Path("dataset.zip")
        if not zip_path.exists():
            zips = sorted(Path(".").glob("*.zip"))
            if not zips:
                raise FileNotFoundError("Place dataset.zip in the current directory.")
            zip_path = zips[0]
            print(f"Found zip: {zip_path}")

    size_mb = zip_path.stat().st_size / (1024 * 1024)
    if size_mb > ZIP_LIMIT_MB:
        raise ValueError(f"Zip file is {size_mb:.1f} MB; limit is {ZIP_LIMIT_MB} MB.")

    print(f"Extracting dataset to {DATA_DIR.resolve()}...")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    data_root = DATA_DIR.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (DATA_DIR / member.filename).resolve()
            if not str(target).startswith(str(data_root)):
                raise ValueError(f"Unsafe zip path found: {member.filename}")
        archive.extractall(DATA_DIR)

    print("Extraction complete.")
    return DATA_DIR


def find_split_dirs(data_path: Path) -> Tuple[Path, Path]:
    train_dir = data_path / "train"
    test_dir = data_path / "test"
    if train_dir.exists() and test_dir.exists():
        return train_dir, test_dir

    for subdir in data_path.iterdir():
        if subdir.is_dir() and (subdir / "train").exists() and (subdir / "test").exists():
            return subdir / "train", subdir / "test"

    raise FileNotFoundError("Could not locate train/ and test/ folders inside the zip.")


def load_dataset(data_path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """Load and preprocess images from train/ and test/ folders."""
    print("[STEP 2] Loading and preprocessing images...")
    train_dir, test_dir = find_split_dirs(data_path)
    class_names = sorted(d.name for d in train_dir.iterdir() if d.is_dir())
    if not class_names:
        raise ValueError("No class folders found in train/.")

    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    print(f"Found {len(class_names)} classes.")

    def process_folder(folder_path: Path, max_images: int) -> Tuple[np.ndarray, np.ndarray]:
        x_list: List[np.ndarray] = []
        y_list: List[int] = []

        for class_name in class_names:
            class_folder = folder_path / class_name
            if not class_folder.exists():
                continue

            files_in_folder = [
                f
                for f in sorted(class_folder.iterdir())
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            ]

            for file_path in files_in_folder:
                if len(x_list) >= max_images:
                    break
                try:
                    with Image.open(file_path) as img:
                        img = img.convert("L")
                        if INVERT_COLORS:
                            img = ImageOps.invert(img)
                        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
                        if APPLY_THRESHOLD:
                            img = img.point(lambda p: 255 if p > 127 else 0)

                        arr = np.asarray(img, dtype=np.float32) / 255.0
                        x_list.append(arr.reshape(-1))
                        y_list.append(class_to_idx[class_name])
                except Exception as exc:
                    print(f"Skipping unreadable image {file_path}: {exc}")

            if len(x_list) >= max_images:
                break

        if not x_list:
            raise ValueError(f"No valid images loaded from {folder_path}.")
        return np.asarray(x_list, dtype=np.float32), np.asarray(y_list, dtype=np.int64)

    print(f"Reading train folder, up to {MAX_TRAIN_IMAGES} images...")
    x_train, y_train = process_folder(train_dir, MAX_TRAIN_IMAGES)
    print(f"Reading test folder, up to {MAX_TEST_IMAGES} images...")
    x_test, y_test = process_folder(test_dir, MAX_TEST_IMAGES)

    print(f"Loaded train={x_train.shape[0]} test={x_test.shape[0]} features={x_train.shape[1]}")
    return x_train, y_train, x_test, y_test, class_names


def get_models() -> Dict[str, Any]:
    """Models constrained for reliable notebook/runtime behavior."""
    common_jobs = max(1, min(2, (os.cpu_count() or 2)))
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=MAX_ITER, solver="saga", n_jobs=common_jobs, random_state=RANDOM_STATE
        ),
        "Ridge Classifier": RidgeClassifier(random_state=RANDOM_STATE),
        "SGD Classifier": SGDClassifier(max_iter=MAX_ITER, tol=1e-3, random_state=RANDOM_STATE),
        "Perceptron": Perceptron(max_iter=MAX_ITER, tol=1e-3, random_state=RANDOM_STATE),
        "Passive Aggressive": PassiveAggressiveClassifier(max_iter=MAX_ITER, tol=1e-3, random_state=RANDOM_STATE),
        "Linear SVC": LinearSVC(max_iter=MAX_ITER, dual="auto", random_state=RANDOM_STATE),
        "Support Vector Classifier (SVC)": SVC(kernel="rbf", cache_size=256, random_state=RANDOM_STATE),
        "K-Neighbors (K=3)": KNeighborsClassifier(n_neighbors=3, n_jobs=common_jobs),
        "K-Neighbors (K=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=common_jobs),
        "Nearest Centroid": NearestCentroid(),
        "Decision Tree": DecisionTreeClassifier(max_depth=30, random_state=RANDOM_STATE),
        "Extra Tree": ExtraTreeClassifier(max_depth=30, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=40, max_depth=30, n_jobs=common_jobs, random_state=RANDOM_STATE
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=40, max_depth=30, n_jobs=common_jobs, random_state=RANDOM_STATE
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=25, max_depth=3, random_state=RANDOM_STATE),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=60, random_state=RANDOM_STATE),
        "Bagging Classifier": BaggingClassifier(n_estimators=8, n_jobs=common_jobs, random_state=RANDOM_STATE),
        "Gaussian Naive Bayes": GaussianNB(),
        "Bernoulli Naive Bayes": BernoulliNB(),
        "Complement Naive Bayes": ComplementNB(),
        "Multinomial Naive Bayes": MultinomialNB(),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=120, early_stopping=True, random_state=RANDOM_STATE
        ),
        "Linear Discriminant Analysis": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
        "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(reg_param=0.1),
        "Dummy Baseline": DummyClassifier(strategy="most_frequent"),
    }


def _evaluate_model_worker(
    result_queue: mp.Queue,
    name: str,
    model: Any,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    class_names: List[str],
) -> None:
    """Run one model in a child process so the parent can terminate hangs."""
    try:
        clf = clone(model)

        start_train = time.perf_counter()
        clf.fit(x_train, y_train)
        train_time = time.perf_counter() - start_train

        start_pred = time.perf_counter()
        y_pred = clf.predict(x_test)
        predict_time = time.perf_counter() - start_pred

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="weighted", zero_division=0
        )
        result_queue.put(
            {
                "status": "ok",
                "name": name,
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "precision_weighted": float(precision),
                "recall_weighted": float(recall),
                "f1_weighted": float(f1),
                "train_time": float(train_time),
                "predict_time": float(predict_time),
                "conf_matrix": confusion_matrix(y_test, y_pred).tolist(),
                "class_report": classification_report(
                    y_test,
                    y_pred,
                    labels=list(range(len(class_names))),
                    target_names=class_names,
                    zero_division=0,
                ),
            }
        )
    except Exception:
        result_queue.put(
            {
                "status": "failed",
                "name": name,
                "error": traceback.format_exc(limit=4),
            }
        )


def evaluate_with_timeout(
    name: str,
    model: Any,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    class_names: List[str],
) -> Dict[str, Any]:
    """Evaluate one model with a hard timeout below two minutes."""
    ctx = mp.get_context("fork" if hasattr(os, "fork") else "spawn")
    result_queue: mp.Queue = ctx.Queue(maxsize=1)
    proc = ctx.Process(
        target=_evaluate_model_worker,
        args=(result_queue, name, model, x_train, y_train, x_test, y_test, class_names),
        daemon=True,
    )

    wall_start = time.perf_counter()
    proc.start()
    proc.join(MODEL_TIMEOUT_SECONDS)
    wall_time = time.perf_counter() - wall_start

    if proc.is_alive():
        proc.terminate()
        proc.join(5)
        if proc.is_alive() and hasattr(proc, "kill"):
            proc.kill()
            proc.join(5)
        return {
            "status": "timeout",
            "name": name,
            "wall_time": float(wall_time),
            "error": f"Stopped after {MODEL_TIMEOUT_SECONDS}s timeout.",
        }

    try:
        result = result_queue.get_nowait()
    except queue.Empty:
        result = {
            "status": "failed",
            "name": name,
            "error": f"Model process exited with code {proc.exitcode} and returned no result.",
        }

    result["wall_time"] = float(wall_time)
    return result


def generate_html_report(results: List[Dict[str, Any]], skipped: List[Dict[str, Any]], class_names: List[str]) -> str:
    sorted_results = sorted(results, key=lambda item: item["accuracy"], reverse=True)
    labels = [r["name"] for r in sorted_results]
    accuracies = [round(r["accuracy"] * 100, 2) for r in sorted_results]
    train_times = [round(r["train_time"], 4) for r in sorted_results]
    f1_scores = [round(r["f1_weighted"] * 100, 2) for r in sorted_results]

    rows = []
    for rank, result in enumerate(sorted_results, 1):
        rows.append(
            f"""
            <tr>
                <td>{rank}</td>
                <td>{html.escape(result["name"])}</td>
                <td>{result["accuracy"]:.4f}</td>
                <td>{result["f1_weighted"]:.4f}</td>
                <td>{result["precision_weighted"]:.4f}</td>
                <td>{result["recall_weighted"]:.4f}</td>
                <td>{result["train_time"]:.4f}s</td>
                <td>{result["predict_time"]:.4f}s</td>
                <td><button onclick="showDetails({json.dumps(result["name"])})">Details</button></td>
            </tr>
            """
        )

    skipped_rows = []
    for item in skipped:
        reason = item.get("error", item["status"]).splitlines()[0]
        skipped_rows.append(
            f"<tr><td>{html.escape(item['name'])}</td><td>{html.escape(item['status'])}</td>"
            f"<td>{html.escape(reason)}</td><td>{item.get('wall_time', 0):.2f}s</td></tr>"
        )

    model_data = {
        r["name"]: {
            "name": r["name"],
            "class_report": html.escape(r["class_report"]),
            "confusion_matrix": r["conf_matrix"],
        }
        for r in sorted_results
    }
    best_accuracy = sorted_results[0]["accuracy"] * 100 if sorted_results else 0

    return f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Bengali Character AutoML Safe Report</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    body {{ margin:0; font-family: Arial, sans-serif; background:#101827; color:#edf2f7; }}
    header, main {{ max-width:1200px; margin:0 auto; padding:28px; }}
    header {{ border-bottom:1px solid #263244; }}
    h1 {{ margin:0 0 8px; }}
    .stats {{ display:flex; gap:16px; flex-wrap:wrap; margin-top:20px; }}
    .stat, section {{ background:#172033; border:1px solid #2c3b52; border-radius:8px; padding:18px; }}
    .stat strong {{ display:block; color:#3dd6b5; font-size:28px; }}
    section {{ margin:20px 0; }}
    table {{ width:100%; border-collapse:collapse; font-size:14px; }}
    th, td {{ padding:10px; border-bottom:1px solid #2c3b52; text-align:left; }}
    th {{ color:#93a4ba; }}
    button {{ cursor:pointer; background:#1b8f7d; color:white; border:0; border-radius:6px; padding:7px 10px; }}
    pre {{ white-space:pre-wrap; background:#0b1220; padding:14px; border-radius:8px; overflow:auto; }}
    .charts {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(320px, 1fr)); gap:20px; }}
    .chart-wrap {{ height:360px; }}
  </style>
</head>
<body>
  <header>
    <h1>Bengali Character AutoML Safe Report</h1>
    <p>Each model was run with a hard {MODEL_TIMEOUT_SECONDS}s timeout to prevent hangs.</p>
    <div class="stats">
      <div class="stat"><span>Completed models</span><strong>{len(results)}</strong></div>
      <div class="stat"><span>Timed out / failed</span><strong>{len(skipped)}</strong></div>
      <div class="stat"><span>Best accuracy</span><strong>{best_accuracy:.2f}%</strong></div>
      <div class="stat"><span>Classes</span><strong>{len(class_names)}</strong></div>
    </div>
  </header>
  <main>
    <section class="charts">
      <div class="chart-wrap"><canvas id="accuracyChart"></canvas></div>
      <div class="chart-wrap"><canvas id="speedChart"></canvas></div>
    </section>
    <section>
      <h2>Leaderboard</h2>
      <table>
        <thead>
          <tr><th>#</th><th>Model</th><th>Accuracy</th><th>F1</th><th>Precision</th><th>Recall</th><th>Train</th><th>Predict</th><th></th></tr>
        </thead>
        <tbody>{"".join(rows)}</tbody>
      </table>
    </section>
    <section>
      <h2>Timed Out / Failed Models</h2>
      <table>
        <thead><tr><th>Model</th><th>Status</th><th>Reason</th><th>Wall Time</th></tr></thead>
        <tbody>{"".join(skipped_rows) or "<tr><td colspan='4'>None</td></tr>"}</tbody>
      </table>
    </section>
    <section id="details" style="display:none">
      <h2 id="detailsTitle"></h2>
      <pre id="detailsReport"></pre>
    </section>
  </main>
  <script>
    const labels = {json.dumps(labels)};
    const accuracies = {json.dumps(accuracies)};
    const f1Scores = {json.dumps(f1_scores)};
    const trainTimes = {json.dumps(train_times)};
    const modelData = {json.dumps(model_data)};

    new Chart(document.getElementById("accuracyChart"), {{
      type: "bar",
      data: {{
        labels,
        datasets: [
          {{ label: "Accuracy %", data: accuracies, backgroundColor: "#37c7ad" }},
          {{ label: "F1 %", data: f1Scores, backgroundColor: "#60a5fa" }}
        ]
      }},
      options: {{ responsive:true, maintainAspectRatio:false, scales:{{ y:{{ min:0, max:100 }} }} }}
    }});

    new Chart(document.getElementById("speedChart"), {{
      type: "scatter",
      data: {{
        datasets: [{{
          label: "Accuracy vs train time",
          data: labels.map((label, i) => ({{ x: trainTimes[i], y: accuracies[i], label }})),
          backgroundColor: "#fbbf24"
        }}]
      }},
      options: {{
        responsive:true,
        maintainAspectRatio:false,
        plugins: {{ tooltip: {{ callbacks: {{ label: ctx => `${{ctx.raw.label}}: ${{ctx.raw.y}}%, ${{ctx.raw.x}}s` }} }} }},
        scales: {{ x: {{ title: {{ display:true, text:"Train time (s)" }} }}, y: {{ min:0, max:100 }} }}
      }}
    }});

    function showDetails(name) {{
      const details = document.getElementById("details");
      document.getElementById("detailsTitle").textContent = name;
      document.getElementById("detailsReport").innerHTML = modelData[name].class_report;
      details.style.display = "block";
      details.scrollIntoView({{ behavior: "smooth" }});
    }}
  </script>
</body>
</html>"""


def main() -> None:
    print("=" * 58)
    print("Bengali Handwritten Character AutoML Benchmarker - Safe")
    print("=" * 58)
    print(f"Per-model hard timeout: {MODEL_TIMEOUT_SECONDS}s")

    try:
        data_path = upload_and_extract_zip()
        x_train, y_train, x_test, y_test, class_names = load_dataset(data_path)
    except Exception as exc:
        print(f"Setup failed: {exc}")
        return

    print("Scaling features...")
    # Pixel values are already in [0, 1], so no extra scaler is required.
    x_train_final = x_train
    x_test_final = x_test

    if USE_PCA:
        try:
            print(f"Applying PCA with n_components={PCA_COMPONENTS}...")
            pca = PCA(n_components=PCA_COMPONENTS, svd_solver="full", random_state=RANDOM_STATE)
            x_train_final = pca.fit_transform(x_train_final)
            x_test_final = pca.transform(x_test_final)
            print(f"Features reduced to {x_train_final.shape[1]}.")
        except Exception as exc:
            print(f"PCA failed; continuing with raw pixels. Reason: {exc}")
            x_train_final = x_train
            x_test_final = x_test

    models = get_models()
    results: List[Dict[str, Any]] = []
    skipped: List[Dict[str, Any]] = []

    print("\nStarting benchmark")
    print("-" * 92)
    print(f"{'Classifier Name':<36} | {'Status':<8} | {'Accuracy':<9} | {'Wall Time':<10}")
    print("-" * 92)

    for name, model in models.items():
        result = evaluate_with_timeout(
            name, model, x_train_final, y_train, x_test_final, y_test, class_names
        )
        if result["status"] == "ok":
            results.append(result)
            print(f"{name:<36} | {'ok':<8} | {result['accuracy'] * 100:>7.2f}% | {result['wall_time']:>8.2f}s")
        else:
            skipped.append(result)
            print(f"{name:<36} | {result['status']:<8} | {'-':<9} | {result.get('wall_time', 0):>8.2f}s")

    if not results:
        print("All classifiers failed or timed out. Try lowering IMAGE_SIZE or MAX_TRAIN_IMAGES.")
        return

    report_filename = "bengali_automl_report.html"
    with open(report_filename, "w", encoding="utf-8") as report_file:
        report_file.write(generate_html_report(results, skipped, class_names))

    print("\nBenchmark complete.")
    print(f"Dashboard saved to: {report_filename}")
    if IN_COLAB:
        files.download(report_filename)


if __name__ == "__main__":
    mp.freeze_support()
    main()


Bengali Handwritten Character AutoML Benchmarker - Safe
Per-model hard timeout: 110s
[STEP 1] Upload your dataset zip file...


Saving Dataset.zip to Dataset.zip
Extracting dataset to /content/bengali_character_data...
Extraction complete.
[STEP 2] Loading and preprocessing images...
Found 50 classes.
Reading train folder, up to 12000 images...
Reading test folder, up to 4000 images...
Loaded train=12000 test=3000 features=784
Scaling features...
Applying PCA with n_components=0.95...
Features reduced to 261.

Starting benchmark
--------------------------------------------------------------------------------------------
Classifier Name                      | Status   | Accuracy  | Wall Time 
--------------------------------------------------------------------------------------------
Logistic Regression                  | timeout  | -         |   110.11s
Ridge Classifier                     | ok       |   45.57% |     0.22s
SGD Classifier                       | ok       |   49.47% |     9.42s
Perceptron                           | ok       |   41.90% |     3.56s
Passive Aggressive                   | ok       |

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_base.py:860: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = effective_n_jobs(self.n_jobs)


K-Neighbors (K=3)                    | ok       |   67.97% |     2.06s


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_base.py:860: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = effective_n_jobs(self.n_jobs)


...

Bagging Classifier                   | ok       |   41.87% |    62.77s
Gaussian Naive Bayes                 | ok       |   51.10% |     0.19s
Bernoulli Naive Bayes                | ok       |   41.13% |     0.17s
Complement Naive Bayes               | failed   | -         |     0.04s
Multinomial Naive Bayes              | failed   | -         |     0.03s
Multi-Layer Perceptron (MLP)         | ok       |   62.87% |     4.40s
Linear Discriminant Analysis         | ok       |   50.50% |     0.39s


/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/p

Quadratic Discriminant Analysis      | ok       |   74.17% |     1.31s
Dummy Baseline                       | ok       |    2.00% |     0.04s

Benchmark complete.
Dashboard saved to: bengali_automl_report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>